# Differential Privacy in Federated Learning (FL4HMA)

This notebook demonstrates how to apply **differential privacy (DP)** to the federated
sparse pixel regression framework. We cover:

1. **Local DP** — client-side DP-SGD (per-batch gradient clipping + Gaussian noise)
2. **Global DP** — server-side clipping of client updates + noise on the aggregated model
3. **Combined DP** — both local and global mechanisms running together
4. **Privacy accounting** — tracking cumulative (ε, δ) budgets via Rényi DP
5. **Comparison** — standard FL vs DP-FL in terms of utility and privacy cost

## 1. Setup and Imports

In [ ]:
# Install if needed (uncomment)
# !pip install -e .[examples]
# !pip install "flwr[simulation]>=1.5"

In [ ]:
import sys
import os

sys.path.insert(0, os.path.join(os.getcwd(), "src"))

import numpy as np
import torch
import matplotlib.pyplot as plt

from fl4hma.federation.differential_privacy import (
    DPConfig,
    DPAccountant,
    DPAphroFlowerClient,
    DPFedAvg,
    dp_train_sparse_pixel,
    clip_model_update,
    add_noise_to_parameters,
    run_federated_dp,
)
from fl4hma.federation.federation import run_federated, run_centralised
from fl4hma.training.training import (
    _get_device,
    get_parameters,
    set_parameters,
    train_sparse_pixel,
    evaluate_sparse_pixel,
)
from fl4hma.models.unet import UNetCNN
from fl4hma.data.data import load_aphro_data, build_country_datasets
from fl4hma.data.torch_dataset import StationPatchDataset

print(f"\u2713 Imports successful")
print(f"  Device: {_get_device()}")
print(f"  PyTorch: {torch.__version__}")

## 2. Understanding Differential Privacy in FL

Differential privacy provides a formal guarantee that an adversary cannot determine
whether any individual data point was included in the training set, even with full
access to the trained model.

In federated learning, DP can be applied at two levels:

| Level | Mechanism | Protects Against |
|-------|-----------|------------------|
| **Local DP** | DP-SGD at each client (clip gradients + add noise) | Curious server inspecting model updates |
| **Global DP** | Server clips & noises aggregated model | External adversary observing the final model |

### Key parameters
- **`clip_norm` (C)**: Maximum allowed L2 norm of gradients/updates
- **`noise_multiplier` (σ/C)**: Ratio of noise standard deviation to clip norm
- **`target_delta` (δ)**: Probability of privacy breach; typically ≪ 1/n

## 3. Privacy Accountant Demo

The `DPAccountant` tracks cumulative privacy loss using Rényi Differential Privacy (RDP)
and converts to standard (ε, δ)-DP.

In [ ]:
# Demonstrate how epsilon grows with training steps
noise_multipliers = [0.5, 1.0, 2.0, 4.0]
max_steps = 200

fig, ax = plt.subplots(1, 1, figsize=(8, 5))

for sigma in noise_multipliers:
    accountant = DPAccountant(target_delta=1e-5)
    epsilons = []
    for step in range(1, max_steps + 1):
        accountant.step(noise_multiplier=sigma, sample_rate=0.01)
        epsilons.append(accountant.epsilon)
    ax.plot(range(1, max_steps + 1), epsilons, label=f"σ = {sigma}")

ax.set_xlabel("Training Steps")
ax.set_ylabel("Privacy Budget ε")
ax.set_title("Privacy Budget Growth (δ = 1e-5, sample_rate = 0.01)", fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\u2713 Higher noise_multiplier → slower ε growth → stronger privacy")

## 4. Configuration

Define experiment paths and hyperparameters.

In [ ]:
# === Data Paths (adjust to your setup) ===
TRAIN_PATH = "data/aphrodite/APHRO_MA_TAVE_025deg_V1808_train.nc"
TEST_PATH = "data/aphrodite/APHRO_MA_TAVE_025deg_V1808_test.nc"
OUTPUT_MASK_PATH = "data/station_masks/out_mask.npy"
CENTRALISED_MASK_PATH = "data/station_masks/stat/centralised_mask.npy"

# Country masks for federated clients
COUNTRY_MASKS = {
    "nepal": "data/station_masks/stat/nepal_mask.npy",
    "india": "data/station_masks/stat/india_mask.npy",
    "china": "data/station_masks/stat/china_mask.npy",
}

# === Model / Training Hyperparameters ===
IN_CHANNELS = 3
BASE_FILTERS = 32
PATCH_SIZE = 32
STRIDE = 32
BATCH_SIZE = 16
LR = 0.001
LOCAL_EPOCHS = 1
NUM_ROUNDS = 5

# === DP Hyperparameters ===
CLIP_NORM = 1.0
NOISE_MULTIPLIER = 1.0
TARGET_DELTA = 1e-5

print("\u2713 Configuration set")

## 5. Load Data

Load the APHRODITE datasets. If the data files are not available, this section
shows how the framework would be invoked — you can substitute with your own
xarray DataArrays.

In [ ]:
# Load data (uncomment and adjust paths when data is available)
# da_train, da_test = load_aphro_data(
#     TRAIN_PATH, TEST_PATH, variable="tave",
#     lon_slice=(60, 105), lat_slice=(20, 45),
# )
# print(f"\u2713 Train shape: {da_train.shape}")
# print(f"\u2713 Test shape:  {da_test.shape}")

# For demonstration without real data, create synthetic arrays
import xarray as xr

np.random.seed(42)
nlat, nlon, ntime = 64, 64, 30
lats = np.linspace(25, 35, nlat)
lons = np.linspace(70, 90, nlon)
times = np.arange(ntime)

# Synthetic temperature-like data
temp = np.random.randn(ntime, nlat, nlon).astype(np.float32) * 5 + 15
lat_grid = np.tile(lats, (ntime, nlon, 1)).transpose(0, 2, 1)
lon_grid = np.tile(lons, (ntime, nlat, 1))

ds = xr.Dataset(
    {
        "tave": (["time", "lat", "lon"], temp),
        "lats": (["time", "lat", "lon"], lat_grid),
        "lons": (["time", "lat", "lon"], lon_grid),
    },
    coords={"time": times, "lat": lats, "lon": lons},
)

da_train = ds[["tave", "lats", "lons"]].to_array().fillna(0)
da_test = ds[["tave", "lats", "lons"]].to_array().fillna(0)

# Synthetic masks
os.makedirs("data/dp_demo_masks", exist_ok=True)

# Output mask (land mask)
output_mask = np.ones((nlat, nlon), dtype=int)
np.save("data/dp_demo_masks/output_mask.npy", output_mask)

# Centralised station mask
station_mask = np.zeros((nlat, nlon), dtype=int)
station_positions = np.random.choice(nlat * nlon, size=50, replace=False)
station_mask.flat[station_positions] = 1
np.save("data/dp_demo_masks/centralised_mask.npy", station_mask)

# Per-client masks (split stations)
positions = np.argwhere(station_mask)
perm = np.random.permutation(len(positions))
splits = np.array_split(perm, 3)
client_names = ["client_A", "client_B", "client_C"]
country_masks_demo = {}

for name, indices in zip(client_names, splits):
    m = np.zeros((nlat, nlon), dtype=int)
    for idx in indices:
        r, c = positions[idx]
        m[r, c] = 1
    path = f"data/dp_demo_masks/{name}_mask.npy"
    np.save(path, m)
    country_masks_demo[name] = path

OUTPUT_MASK_DEMO = "data/dp_demo_masks/output_mask.npy"
CENTRALISED_MASK_DEMO = "data/dp_demo_masks/centralised_mask.npy"

print(f"\u2713 Synthetic data created: {da_train.shape}")
print(f"  Clients: {list(country_masks_demo.keys())}")
print(f"  Stations per client: {[np.load(p).sum() for p in country_masks_demo.values()]}")

## 6. Local DP: DP-SGD Training

Local DP protects individual training samples from a **curious server** that
inspects the model updates sent by each client.

The mechanism:
1. Compute gradients normally
2. **Clip** gradient norm to `C`
3. **Add** Gaussian noise with σ = `noise_multiplier × C`
4. Update model parameters

In [ ]:
from torch.utils.data import DataLoader

# Create a dataset for one client
client_ds = StationPatchDataset(
    da_train,
    input_mask_path=list(country_masks_demo.values())[0],
    output_mask_path=OUTPUT_MASK_DEMO,
    patch_size=PATCH_SIZE,
    stride=STRIDE,
)
client_loader = DataLoader(client_ds, batch_size=BATCH_SIZE, shuffle=True)
print(f"Client dataset: {len(client_ds)} patches")

# Train without DP
model_no_dp = UNetCNN(in_channels=IN_CHANNELS, out_channels=1, base_filters=BASE_FILTERS)
loss_no_dp = train_sparse_pixel(model_no_dp, client_loader, epochs=3, lr=LR)
print(f"\nNo DP - Final loss: {loss_no_dp:.4f}")

# Train with local DP
model_dp = UNetCNN(in_channels=IN_CHANNELS, out_channels=1, base_filters=BASE_FILTERS)
dp_cfg = DPConfig(
    clip_norm=CLIP_NORM,
    noise_multiplier=NOISE_MULTIPLIER,
    target_delta=TARGET_DELTA,
    local_dp=True,
    global_dp=False,
)
loss_dp, accountant = dp_train_sparse_pixel(
    model_dp, client_loader, dp_config=dp_cfg, epochs=3, lr=LR,
)
print(f"Local DP - Final loss: {loss_dp:.4f}")
print(f"  Privacy spent: ε = {accountant.epsilon:.4f}, δ = {TARGET_DELTA}")
print(f"  Steps recorded: {accountant.steps}")

## 7. Global DP: Server-side Noise

Global DP protects each **client's contribution** from being extracted from
the published global model. The server:

1. Receives model updates from each client
2. **Clips** each update Δ to have ‖Δ‖₂ ≤ C
3. Aggregates (averages) the clipped updates
4. **Adds** Gaussian noise σ = noise_multiplier × C / num_clients

In [ ]:
# Demonstrate clipping and noise on model parameters
model_a = UNetCNN(in_channels=IN_CHANNELS, out_channels=1, base_filters=BASE_FILTERS)
model_b = UNetCNN(in_channels=IN_CHANNELS, out_channels=1, base_filters=BASE_FILTERS)

# Simulate: model_b is the "updated" version after local training
# (train a few steps to create a realistic delta)
set_parameters(model_b, get_parameters(model_a))  # same starting point
_ = train_sparse_pixel(model_b, client_loader, epochs=2, lr=LR)

original_params = get_parameters(model_a)
updated_params = get_parameters(model_b)

# Compute update norm before clipping
delta = np.concatenate([(u - o).ravel() for u, o in zip(updated_params, original_params)])
print(f"Update L2 norm (before clipping): {np.linalg.norm(delta):.4f}")

# Clip the update
clipped_params = clip_model_update(original_params, updated_params, clip_norm=1.0)
delta_clipped = np.concatenate([(c - o).ravel() for c, o in zip(clipped_params, original_params)])
print(f"Update L2 norm (after clipping):  {np.linalg.norm(delta_clipped):.4f}")

# Add noise
noise_std = NOISE_MULTIPLIER * CLIP_NORM / 3  # 3 clients
noisy_params = add_noise_to_parameters(clipped_params, noise_std=noise_std)
noise_applied = np.concatenate([(n - c).ravel() for n, c in zip(noisy_params, clipped_params)])
print(f"Noise L2 norm applied:            {np.linalg.norm(noise_applied):.4f}")
print(f"Noise σ per parameter:            {noise_std:.6f}")

## 8. DP Configuration Variants

The `DPConfig` dataclass controls which DP mechanisms are active.

In [ ]:
# Local DP only
cfg_local = DPConfig(
    clip_norm=1.0,
    noise_multiplier=1.0,
    target_delta=1e-5,
    local_dp=True,
    global_dp=False,
)

# Global DP only
cfg_global = DPConfig(
    clip_norm=1.0,
    noise_multiplier=1.0,
    target_delta=1e-5,
    local_dp=False,
    global_dp=True,
)

# Both local + global DP
cfg_both = DPConfig(
    clip_norm=1.0,
    noise_multiplier=1.0,
    target_delta=1e-5,
    local_dp=True,
    global_dp=True,
)

# Aggressive privacy (higher noise)
cfg_strong = DPConfig(
    clip_norm=0.5,
    noise_multiplier=2.0,
    target_delta=1e-6,
    local_dp=True,
    global_dp=True,
)

print("DP Configurations:")
for name, cfg in [("Local only", cfg_local), ("Global only", cfg_global),
                  ("Both", cfg_both), ("Strong", cfg_strong)]:
    print(f"  {name:12s}: C={cfg.clip_norm}, σ={cfg.noise_multiplier}, "
          f"δ={cfg.target_delta}, local={cfg.local_dp}, global={cfg.global_dp}")

## 9. Run Federated Learning with DP

Use `run_federated_dp()` — the drop-in replacement for `run_federated()` with
DP mechanisms enabled.

In [ ]:
%%time

# Run with combined local + global DP
dp_results = run_federated_dp(
    da_train=da_train,
    da_test=da_test,
    country_masks=country_masks_demo,
    output_mask_path=OUTPUT_MASK_DEMO,
    centralised_mask_path=CENTRALISED_MASK_DEMO,
    dp_config=cfg_both,
    num_rounds=NUM_ROUNDS,
    local_epochs=LOCAL_EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    in_channels=IN_CHANNELS,
    base_filters=BASE_FILTERS,
    patch_size=PATCH_SIZE,
    stride=STRIDE,
)

print(f"\n\u2713 DP Federated complete")
print(f"  Final MSE:  {dp_results['final_mse']:.6f}")
print(f"  Final RMSE: {dp_results['final_rmse']:.6f}")
print(f"  Global ε:   {dp_results['global_epsilon']:.4f}")

## 10. Comparison: Standard FL vs DP-FL

Run standard federated learning (no DP) as a baseline.

In [ ]:
%%time

# Standard federated (no DP)
std_results = run_federated(
    da_train=da_train,
    da_test=da_test,
    country_masks=country_masks_demo,
    output_mask_path=OUTPUT_MASK_DEMO,
    centralised_mask_path=CENTRALISED_MASK_DEMO,
    num_rounds=NUM_ROUNDS,
    local_epochs=LOCAL_EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    in_channels=IN_CHANNELS,
    base_filters=BASE_FILTERS,
    patch_size=PATCH_SIZE,
    stride=STRIDE,
)

print(f"\n\u2713 Standard Federated complete")
print(f"  Final MSE:  {std_results['final_mse']:.6f}")
print(f"  Final RMSE: {std_results['final_rmse']:.6f}")

In [ ]:
# Visualise comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Loss curves
ax = axes[0]
ax.plot(std_results["rounds"], std_results["losses"], "o-", label="Standard FL")
ax.plot(dp_results["rounds"], dp_results["losses"], "s--", label="DP-FL")
ax.set_xlabel("Round")
ax.set_ylabel("Loss")
ax.set_title("Server-side Loss", fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)

# MSE curves
ax = axes[1]
if std_results["mse_values"]:
    ax.plot(std_results["rounds"][:len(std_results["mse_values"])],
            std_results["mse_values"], "o-", label="Standard FL")
if dp_results["mse_values"]:
    ax.plot(dp_results["rounds"][:len(dp_results["mse_values"])],
            dp_results["mse_values"], "s--", label="DP-FL")
ax.set_xlabel("Round")
ax.set_ylabel("MSE")
ax.set_title("Server-side MSE", fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle("Standard FL vs DP-FL", fontweight="bold", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 11. Privacy–Utility Trade-off

Sweep across different noise multipliers to visualise the trade-off between
privacy (lower ε is better) and utility (lower MSE is better).

In [ ]:
# Sweep noise multipliers (local DP only for speed)
noise_sweep = [0.1, 0.5, 1.0, 2.0, 4.0]
sweep_results = []

for sigma in noise_sweep:
    print(f"Training with noise_multiplier = {sigma}...")
    model_sweep = UNetCNN(in_channels=IN_CHANNELS, out_channels=1, base_filters=BASE_FILTERS)
    cfg = DPConfig(
        clip_norm=CLIP_NORM,
        noise_multiplier=sigma,
        target_delta=TARGET_DELTA,
        local_dp=True,
        global_dp=False,
    )
    loss, acct = dp_train_sparse_pixel(
        model_sweep, client_loader, dp_config=cfg, epochs=3, lr=LR,
    )
    # Evaluate
    metrics = evaluate_sparse_pixel(model_sweep, client_loader)
    sweep_results.append({
        "noise_multiplier": sigma,
        "epsilon": acct.epsilon,
        "mse": metrics["mse"],
        "loss": loss,
    })
    print(f"  ε = {acct.epsilon:.4f}, MSE = {metrics['mse']:.6f}")

print("\n\u2713 Sweep complete")

In [ ]:
# Plot privacy-utility trade-off
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

epsilons = [r["epsilon"] for r in sweep_results]
mses = [r["mse"] for r in sweep_results]
sigmas = [r["noise_multiplier"] for r in sweep_results]

# ε vs MSE
ax = axes[0]
ax.scatter(epsilons, mses, s=100, c=sigmas, cmap="viridis", edgecolors="k", zorder=5)
for i, sigma in enumerate(sigmas):
    ax.annotate(f"σ={sigma}", (epsilons[i], mses[i]),
                textcoords="offset points", xytext=(5, 5), fontsize=9)
ax.set_xlabel("Privacy Budget ε (lower = more private)")
ax.set_ylabel("MSE (lower = better utility)")
ax.set_title("Privacy–Utility Trade-off", fontweight="bold")
ax.grid(True, alpha=0.3)

# Noise multiplier vs both
ax = axes[1]
ax2 = ax.twinx()
l1 = ax.bar(range(len(sigmas)), epsilons, alpha=0.6, color="steelblue", label="ε")
l2 = ax2.plot(range(len(sigmas)), mses, "ro-", label="MSE", linewidth=2)
ax.set_xticks(range(len(sigmas)))
ax.set_xticklabels([f"σ={s}" for s in sigmas])
ax.set_ylabel("ε (privacy budget)", color="steelblue")
ax2.set_ylabel("MSE", color="red")
ax.set_title("Effect of Noise Multiplier", fontweight="bold")
ax.legend(loc="upper left")
ax2.legend(loc="upper right")

plt.tight_layout()
plt.show()

## 12. Summary

| Aspect | Description |
|--------|-------------|
| **Local DP** | DP-SGD at each client: gradient clipping + Gaussian noise |
| **Global DP** | Server clips client updates + adds noise to aggregate |
| **Accounting** | Rényi DP with conversion to (ε, δ)-DP |
| **Integration** | Drop-in `DPConfig` + `run_federated_dp()` replaces `run_federated()` |
| **Trade-off** | Higher σ → stronger privacy (lower ε) but higher MSE |

### Key Takeaways

- **Local DP** protects individual samples from a curious server
- **Global DP** protects individual clients from model inversion attacks
- The `noise_multiplier` is the primary knob for the privacy–utility trade-off
- Privacy budget ε accumulates over training steps/rounds — fewer rounds = less budget spent
- The `DPAccountant` provides real-time ε tracking for informed stopping decisions
- Both mechanisms can be used independently or combined for defence-in-depth